In [1]:
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.cosmology import Planck15
import astropy.cosmology as cosmo
import astropy.units as u
from scipy.optimize import minimize
from tqdm import tqdm
from astropy.io import fits
import pickle
from scipy import integrate
from sklearn.neighbors import KernelDensity


H0 =  73.8
Om = 0.3
Ok = 0.0
w = -1.0
zs = np.linspace(0.001, 5, 100)
zs = zs[1:] # We don't want z = 0

my_cosmo = cosmo.wCDM(H0=H0, Om0=Om, Ode0=1.0-Om-Ok)
dls_true = my_cosmo.luminosity_distance(zs).to(u.Mpc).value
ys = zs/(1+zs)

In [2]:
z_pantheon = np.lib.format.open_memmap('/home/jessica/Jess/Projects/Ongoing/Cosmography/Python/Colab/Pantheon_like.npy')

In [4]:
def sigma_dl(dL):
    
    sigma_dl = dL * 0.22 * np.log(10)/5
    return sigma_dl

def chi_square(theta,func, z, data, order):
    sigma = sigma_dl(data)
    diff2 = (func(z, *theta, order=order) - data)**2

    chi = np.sum(diff2/sigma**2)
    return chi

# Loop in z and order

In [ ]:
# Function to handle minimization for each n iteration
def minimize_for_n(args):
    idx, zmax, order, pade, zmin, met, fun,i = args
    #print(idx,zmax,order,pade,zmin,met,fun,i)
    # Generate true values
    np.random.seed()
    success=[]
    H0_true = np.random.uniform(65, 75)

    Om_true = np.random.uniform(0.2, 0.4)

    if K:
                Ok_true = np.random.uniform(-0.01, 0.01)
    else:
                Ok_true = 0
    if w:
                w_true = np.random.uniform(-1.5,-0.5)
    else:
                w_true = -1
    pars = to_cosmographic(H0_true, Om_true, Ok_true, w_true)

    true_vals = np.full(order, np.nan)


    # Compute redshifts
    for j in range(order):
      true_vals[j] = pars[j]
    zs = np.arange(zmin, zmax, 0.005)
    #zs=np.linspace(zmin,zmax,1000)
    my_cosmo = cosmo.wCDM(H0=H0_true, Om0=Om_true, Ode0=1.0 - Om_true - Ok_true, w0=w_true)
    dls_true = my_cosmo.luminosity_distance(zs).to(u.Mpc).value

    # Minimizer parameters
    start = true_vals + abs(true_vals) * np.random.normal(0, 0.05, order)
    lower_bounds = [60, -1, -2, -2, -10, None, -0.1]
    upper_bounds = [80, 1, 2, 2, 10, None, 0.1]
    bounds = [(lower_bounds[k], upper_bounds[k]) for k in range(order)]

    # Results storage
    # min_z = np.full(order, np.nan)
    # min_y = np.full(order, np.nan)
    # min_log = np.full(order, np.nan)
    min_pade = np.full((len(pade), order), np.nan)
    success_flags = np.full(3 + len(pade), np.nan)

    # Minimizing for different series
    result_z = minimize(fun, start, args=(z_series, zs, dls_true, order), bounds=bounds, method=met, options={'maxiter': 200000}, tol=1e-12)
    if result_z.success:
        min_z_order[idx, :] = result_z.x
    ys = zs / (1 + zs)
    result_y = minimize(fun, start, args=(y_series, ys, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
    if result_y.success:
        min_y_order[idx, :] = result_y.x


    log1zs = np.log(1 + zs)
    result_log = minimize(fun, start, args=(log_series, log1zs, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
    if result_log.success:
        min_log_order[idx, :] = result_log.x
    success_pade = []
    # Minimizing for Pade functions
    pade_list=[]
    if len(pade) != 0:
        for j in range(len(pade)):
            result_pade = minimize(fun, start, args=(pade[j], zs, dls_true, 0), method=met, bounds=bounds, options={'maxiter':200000}, tol=1e-12)
            #print(result_pade)
            if result_pade.success:
                pade_list.append(result_pade.x)
            success_pade.append(result_pade.success)
    
    success.append(result_z.success)
    success.append(result_y.success)
    success.append(result_log.success)

    success.extend(success_pade)

    #success_flags[:] = [result_z.success, result_y.success, result_log.success] + success_pade

    return (i,result_z.x, result_y.x,result_log.x,pade_list,np.append(true_vals,[Ok_true,w_true,Om_true]),success)

if __name__ == "__main__":
    # Define number of processes
    num_processes = min(cpu_count(), n)
    print('Number of CPUS avaliable:', num_processes)
    # Define the range of zmax values and prepare arguments
    #z_max_array = np.linspace(0.2, 2, N)
    z_max_array=np.linspace(1,9,N)
    order_list = [3, 4, 5, 6]

    # Collecting results
    min_z_awkward = []
    min_y_awkward = []
    min_log_awkward = []
    min_pade_awkward = []
    true_awkward = []
    success_awkward = []

    for ord, order in enumerate(order_list):
        # Choose functions based on parameters
        if order < 3:
            pade = []
        elif order == 3:
            pade = [dLpade21]
        elif order == 4:
            pade = [dLpade21, dLpade22]
        else:
            pade = [dLpade21, dLpade22, dLpade32]

        # Arguments for each parallel task over n

        # Initialize storage for current order
        # with Pool(num_processes) as pool:
        #     results = list(tqdm(pool.imap(minimize_for_n, args_list), total=len(args_list)))

        # Store results for each z_max and n
        min_z_order = np.full((len(z_max_array),n, order), np.nan)
        min_y_order = np.full((len(z_max_array),n, order), np.nan)
        min_log_order = np.full((len(z_max_array),n, order), np.nan)
        min_pade_order = np.full((len(z_max_array), len(pade),n, order), np.nan)
        true_order = np.full((len(z_max_array),n, order+3), np.nan)
        success_order = np.full((len(z_max_array), n,3 + len(pade)), np.nan)
        for idx, zmax in enumerate(z_max_array):
            args_list = [(idx, zmax, order, pade, zmin, met, eval(fun.__name__),i) for i in range(n)]

            with Pool(num_processes) as pool:
                results = list(pool.imap(minimize_for_n, args_list))
            for i, result_z, result_y,result_log,pade_list,truth,success in results:
              min_z_order[idx, i, :] =  result_z
              min_y_order[idx, i, :] = result_y
              min_log_order[idx, i, :] = result_log
              for j in range(order+2):
                true_order[idx, i, j] = truth[j]
              if len(pade)!=0:
                j=0
                for pade_result in pade_list:
                  min_pade_order[idx, j, i, :] = pade_result
                  j+=1
              success_order[idx, i, :] = success


            # Fill in results
            # for i in range(n):
            #     res = results[idx * n + i]
            #     min_z_order[idx,i, :] = res[0]
            #     min_y_order[idx,i, :] = res[1]
            #     min_log_order[idx,i, :] = res[2]
            #     min_pade_order[idx,i, :, :] = res[3]
            #     true_order[idx,i, :] = res[4]
            #     success_order[idx,i, :] = res[5]

            # Append current order results to lists
            min_z_awkward.append(ak.Array(min_z_order))
            min_y_awkward.append(ak.Array(min_y_order))
            min_log_awkward.append(ak.Array(min_log_order))
            min_pade_awkward.append(ak.Array(min_pade_order))
            true_awkward.append(ak.Array(true_order))
            success_awkward.append(ak.Array(success_order))
            # success.append(success_order)

            # true.append(true_order)
            # success.append(success_order)

min_z_awkward = ak.Array(min_z_awkward)
min_y_awkward = ak.Array(min_y_awkward)
min_log_awkward = ak.Array(min_log_awkward)
min_pade_awkward = ak.Array(min_pade_awkward)
true_awkward = ak.Array(true_awkward)
success_awkward = ak.Array(success_awkward)

In [ ]:
# it is working!

n = 1000              # number of runs
N = 1               # number of z_max we want
K = False            
w = False
MET = 4
fun = chi_square

# Minimizer method selection
if MET==1:
    met='Nelder-Mead' #very slow
# elif MET==2:
#     met='SLSQP'
# elif MET==3:
#     met='Powell' # ultra slow
elif MET==4:
    met="trust-constr"
elif MET==5:
    met="TNC"
elif MET ==6:
    met="COBYLA"
elif MET ==7:
    met="L-BFGS-B" #fast
# elif MET==8:
#     met="Newton-CG"
# elif MET==9:
#     met='BFGS'
# elif MET==10:
#     met='CG'
# elif MET==11:
#     met='trust-exact'

# Define the range of zmax values
z_max_array = [1]
order_list = [2, 3, 4, 5]

# Initialize lists to store results
min_z = []
min_y = []
min_log = []
min_pade = []
true = []
success = []
chisquare = []

   

for ord, order in enumerate(order_list):
    # Choose functions based on parameters
    if order < 3:
        pade = []
    elif order == 3:
        pade = [dLpade21]
    elif order == 4:
        pade = [dLpade21, dLpade22]
    elif order == 5:
        pade = [dLpade21, dLpade22, dLpade32, dLpadeaverage]
    else:
        pade = [dLpade21, dLpade22, dLpade32, dLpadeaverage, dLpade42, dLpade51]

    # Initialize storage for current order
    min_z_order = np.full((len(z_max_array), n, order), np.nan)
    min_y_order = np.full((len(z_max_array), n, order), np.nan)
    min_log_order = np.full((len(z_max_array), n, order), np.nan)
    min_pade_order = np.full((len(z_max_array), len(pade), n, order), np.nan)
    true_order = np.full((len(z_max_array), n, order), np.nan)
    success_order = np.full((len(z_max_array), n, 3 + len(pade)), np.nan)
    chisq_order = np.full((len(z_max_array), n, 3 + len(pade)), np.nan)

    # Loop over z_max_array
    for idx, zmax in enumerate(tqdm(z_max_array)):
        for i in range(n):
            H0_true = np.random.uniform(65, 75)
            Om_true = np.random.uniform(0.2, 0.4)
            
            if K:
                Ok_true = np.random.uniform(-0.05, 0.05)
            else:
                Ok_true = 0

            if w:
                w_true = np.random.uniform(-1.5,-0.5)
            else:
                w_true = -1
                
            results_pade = []

            pars = to_cosmographic(H0_true, Om_true, Ok_true, w_true)
            for j in range(order):
                true_order[idx, i, j] = pars[j]

            zs = z_pantheon
            my_cosmo = cosmo.wCDM(H0=H0_true, Om0=Om_true, Ode0=1.0 - Om_true - Ok_true, w0=w_true)
            dls_true = my_cosmo.luminosity_distance(zs).to(u.Mpc).value

            # Minimizer parameters
            start = true_order[idx, i] + abs(true_order[idx, i]) * np.random.normal(0, 0.05, order)
            if w:
                lower_bounds = [60, -2.5,  -1, -2.5,    -5,  -200]
                upper_bounds = [80,    1,   15,  60,  350, 2600]

            else:
                lower_bounds = [60, -1, -0.5, -2.5, -5, -45 ]
                upper_bounds = [80,  1,  1.5,     2, 15   , 5]
 
            bounds = [(lower_bounds[k], upper_bounds[k]) for k in range(order)]


            # Minimizing for different series
            result_z = minimize(fun, start, args=(z_series, zs, dls_true, order), bounds=bounds, method=met, options={'maxiter': 200000}, tol=1e-12)
            if result_z.success == True:
                min_z_order[idx, i, :] = result_z.x
                chisq_order[idx, i, 0] = chi_square(result_z.x,z_series, zs, dls_true, order=order)/(len(zs) - order)

            ys = zs / (1 + zs)
            result_y = minimize(fun, start, args=(y_series, ys, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
            if result_y.success == True:
                min_y_order[idx, i, :] = result_y.x
                chisq_order[idx, i, 1] = chi_square(result_y.x,y_series, zs, dls_true, order=order)/(len(zs) - order)
    
            log1zs = np.log(1 + zs)
            result_log = minimize(fun, start, args=(log_series, log1zs, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
            if result_log.success == True:
                min_log_order[idx, i, :] = result_log.x
                chisq_order[idx, i, 2] = chi_square(result_log.x,log_series, zs, dls_true, order=order)/(len(zs) - order)
    
            # Minimizing for Pade functions
            success_pade = []
            #message_pade = []
            if len(pade) !=0:
                for j in range(len(pade)):
                    result_pade = minimize(fun, start, args=(pade[j], zs, dls_true, 0), method=met, bounds=bounds, options={'maxiter':200000}, tol=1e-12)
                    #results_pade.append(result_pade)
                    if result_pade.success == True:
                        min_pade_order[idx, j, i, :] = result_pade.x
                        chisq_order[idx, i, 3 + j] = chi_square(result_pade.x,pade[j], zs, dls_true, order =0)/(len(zs) - order)
                    success_pade.append(result_pade.success)
        
            success_order[idx, i, :] = result_z.success, result_y.success, result_log.success, *success_pade
                

    # Append current order results to lists
    min_z.append(min_z_order)
    min_y.append(min_y_order)
    min_log.append(min_log_order)
    min_pade.append(min_pade_order)
    true.append(true_order)
    success.append(success_order)
    chisquare.append(chisq_order)

# Generate files

In [ ]:
import awkward as ak

min_z_awkward = ak.Array(min_z)
min_y_awkward = ak.Array(min_y)
min_log_awkward = ak.Array(min_log)
min_pade_awkward = ak.Array(min_pade)
true_awkward = ak.Array(true)
success_awkward = ak.Array(success)
chisquare_awkward = ak.Array(chisquare)



stri='Pantheon_MET4_sim1000'
ak.to_parquet(min_z_awkward, f"min_z_awkward_{stri}.parquet")
ak.to_parquet(min_y_awkward, f"min_y_awkward_{stri}.parquet")
ak.to_parquet(min_log_awkward, f'min_log_awkward_{stri}.parquet')
ak.to_parquet(min_pade_awkward, f'min_pade_awkward_{stri}.parquet')
ak.to_parquet(true_awkward, f'true_awkward_{stri}.parquet')
ak.to_parquet(success_awkward, f'success_awkward_{stri}.parquet')
ak.to_parquet(chisquare_awkward, f'chisquare_awkward_{stri}.parquet')

# Fixed order, z loop

In [20]:
# #  Loop over z is working. Adjust z_max_array as you wish.
# #  Observe that the order is 4 now. The loop with the order is below.

# # Simulation parameters:
# np.random.seed(27)

# n = 5                 # number of runs
# K = False            
# w = False
# pen = False            # chi_square with or without penalty
# order = 4
# zmin = 0.0008
# MET = 4

# # Define the range of zmax values
# z_max_array = np.linspace(0.2, 2, 2)

# # Minimizer method selection
# if MET==1:
#     met='Nelder-Mead'
# # elif MET==2:
# #     met='SLSQP'
# # elif MET==3:
# #     met='Powell'
# elif MET==4:
#     met="trust-constr"
# elif MET==5:
#     met="TNC"
# elif MET ==6:
#     met="COBYLA"
# elif MET ==7:
#     met="L-BFGS-B"
# # elif MET==8:
# #     met="Newton-CG"
# # elif MET==9:
# #     met='BFGS'
# # elif MET==10:
# #     met='CG'
# # elif MET==11:
# #     met='trust-exact'


# # Choose functions based on parameters
# if order < 3:
#     pade = []
# elif order == 3:
#     pade = [dLpade21]
# elif order == 4: 
#     pade = [dLpade21, dLpade22, dLpade32]
# else:
#     pade = [dLpade21, dLpade22, dLpade32, dLpadeaverage, dLpade42, dLpade51]
    
# if pen == False:
#     fun = chi_square
# else:
#     fun = chi_square_with_penalty
    
# # Initialize storage for results
# min_z = np.zeros((len(z_max_array), n, order))
# min_y = np.zeros((len(z_max_array), n, order))
# min_log = np.zeros((len(z_max_array), n, order))
# min_pade = np.zeros((len(z_max_array), len(pade), n, order))
# true = np.zeros((len(z_max_array), n, order))
# success = np.zeros((len(z_max_array), n, 3+len(pade)))
    
# # Loop over z_max_array
# for idx, zmax in enumerate(tqdm(z_max_array)):
#     for i in range(n):
#         H0_true = np.random.uniform(65, 75)
#         Om_true = np.random.uniform(0.2, 0.4)
#         Ok_true = 0
#         w_true = -1
#         results_pade = []
    
#         pars = to_cosmographic(H0_true, Om_true, Ok_true, w_true)
#         for j in range(order):
#             true[idx][i][j] = pars[j]
    
#         zs = np.arange(zmin, zmax, 0.1)
#         my_cosmo = cosmo.wCDM(H0=H0_true, Om0=Om_true, Ode0=1.0-Om_true-Ok_true, w0=w_true)
#         dls_true = my_cosmo.luminosity_distance(zs).to(u.Mpc).value
    
#         # Minimizer parameters
#         start = true[idx][i] + abs(true[idx][i]) * np.random.normal(0, 0.05, order)
#         # H0, q0, j0, s0, c0, p0, k
#         lower_bounds = [60, -1, -2, -2, -10, None, -0.1]
#         upper_bounds = [80, 1, 2, 2, 10, None, 0.1]
#         bounds = [(lower_bounds[i], upper_bounds[i]) for i in range(order)]
    
#         # Minimizing for different series
#         result_z = minimize(fun, start, args=(z_series, zs, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
#         min_z[idx, i, :] = result_z.x
    
#         ys = zs / (1 + zs)
#         result_y = minimize(fun, start, args=(y_series, ys, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
#         min_y[idx, i, :] = result_y.x
    
#         log1zs = np.log(1 + zs)
#         result_log = minimize(fun, start, args=(log_series, log1zs, dls_true, order), bounds=bounds, method=met, options={'maxiter':200000}, tol=1e-12)
#         min_log[idx, i, :] = result_log.x
    
#         # Minimizing for Pade functions
#         success_pade = []
#         for j in range(len(pade)):
#             result_pade = minimize(fun, start, args=(pade[j], zs, dls_true, 0), method=met, bounds=bounds, options={'maxiter':200000}, tol=1e-12)
#             results_pade.append(result_pade)
#             min_pade[idx, j, i, :] = result_pade.x
#             success_pade.append(result_pade.success)
    
#         success[idx, i, :] = result_z.success, result_y.success, result_log.success, *success_pade    

  0%|                                                     | 0/2 [00:00<?, ?it/s]/home/jessica/miniconda3/envs/py3/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:231: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)
  0%|                                                     | 0/2 [00:07<?, ?it/s]


KeyboardInterrupt: 